<a href="https://colab.research.google.com/github/Omar-Anwar-Dev/news-category-classification/blob/main/notebooks/00_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# News Category Classification — Main Notebook

Multi-class news classifier (42 categories) using headline + short description.

**Pipeline:** Kaggle dataset → preprocessing → TF-IDF + numeric features → 6 classical models + RoBERTa-embedding SVM → Gradio UI with Groq LLM explanations and similarity retrieval.

Per ADR-009, all pipeline code lives in this notebook. The only Python file maintained outside is `src/preprocessing.py` (because its `clean_text()` is unit-tested in CI).

Section headers map directly to sprint-1 tasks; later sprints fill in the remaining models and the full Gradio UI.

## Colab Quick-Start (run-once cell below)

If you opened this notebook on Colab, run the cell below first. It clones
the repo, installs dependencies, downloads NLTK corpora, and pulls Groq +
Kaggle credentials from the **Colab Secrets** panel (key icon on the left
sidebar) so the rest of the notebook runs unchanged.

If you opened this on your laptop with a local venv, the cell is a no-op —
your existing setup keeps working.

**Required Colab Secrets** (sidebar → key icon → "+ Add new secret",
toggle "Notebook access" on for each):
- `GROQ_API_KEY` — from https://console.groq.com/keys
- `KAGGLE_USERNAME` and `KAGGLE_KEY` — open `kaggle.json` in any text editor; copy `"username"` and `"key"` values

In [ ]:
"""Colab one-time bootstrap — safe to re-run; safe on local Windows/Linux."""

import json
import os
import sys

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    REPO_DIR = "/content/news-category-classification"
    REPO_URL = "https://github.com/Omar-Anwar-Dev/news-category-classification.git"

    if not os.path.exists(REPO_DIR):
        print(f"Cloning {REPO_URL} ...")
        os.system(f"git clone -q {REPO_URL} {REPO_DIR}")
    os.chdir(REPO_DIR)

    print("Installing dependencies (first run: ~3 min)...")
    os.system("pip install -q -r requirements.txt")

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    import nltk

    for r in ["punkt", "punkt_tab", "stopwords", "wordnet"]:
        nltk.download(r, quiet=True)

    # --- Groq key from Colab Secrets ---
    try:
        from google.colab import userdata

        if not os.environ.get("GROQ_API_KEY"):
            try:
                os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
                print("OK: GROQ_API_KEY loaded from Colab Secrets")
            except Exception:
                print("WARN: GROQ_API_KEY not found in Colab Secrets — Groq cell will skip.")
    except ImportError:
        pass

    # --- Kaggle credentials from Colab Secrets (write kaggle.json) ---
    kaggle_json = os.path.join(REPO_DIR, "kaggle.json")
    if not os.path.exists(kaggle_json):
        try:
            from google.colab import userdata

            creds = {
                "username": userdata.get("KAGGLE_USERNAME"),
                "key": userdata.get("KAGGLE_KEY"),
            }
            with open(kaggle_json, "w") as f:
                json.dump(creds, f)
            os.chmod(kaggle_json, 0o600)
            print("OK: kaggle.json written from Colab Secrets")
        except Exception:
            print(
                "WARN: Kaggle credentials not in Colab Secrets. Either add "
                "KAGGLE_USERNAME and KAGGLE_KEY to Secrets, or upload kaggle.json "
                "manually via the Files panel on the left."
            )

    print("\nColab bootstrap complete.")
else:
    print("Running locally — bootstrap skipped (assumes venv + .env are configured).")

## Setup & Imports

Runs once at the top of the notebook. Idempotent — safe to re-run.

In [ ]:
import logging
import os
import sys
from pathlib import Path

# Make `src/` importable when running from the notebooks/ folder.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Load .env if present (local Windows / Linux). On Colab use Colab Secrets UI instead.
try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")
except ImportError:
    pass

# Standard project paths.
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EMBEDDINGS_DIR = DATA_DIR / "embeddings"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
CONFUSION_DIR = REPORTS_DIR / "confusion"
for d in (RAW_DIR, PROCESSED_DIR, EMBEDDINGS_DIR, MODELS_DIR, REPORTS_DIR, CONFUSION_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Single source of truth for randomness (ADR-section: reproducibility).
RANDOM_STATE = 42

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("main")
log.info(f"Project root: {PROJECT_ROOT}")

## 1. Data Loading — S1-T5

Download the News Category Dataset (~209K rows) from Kaggle. The first run hits Kaggle; subsequent runs read from local cache under `data/raw/`.

In [ ]:
"""S1-T5 — Kaggle download with local cache + JSON-Lines parse.

Returns a DataFrame with columns: link, headline, category,
short_description, authors, date. About 209K rows (full file ~80 MB).
First call hits Kaggle; subsequent calls read from `data/raw/`.
"""

import json

import pandas as pd

DATASET_SLUG = "rmisra/news-category-dataset"
DATASET_FILENAME = "News_Category_Dataset_v3.json"


def load_dataset(force_download: bool = False) -> pd.DataFrame:
    cache_path = RAW_DIR / DATASET_FILENAME
    if cache_path.exists() and not force_download:
        log.info(f"Using cached dataset at {cache_path} ({cache_path.stat().st_size / 1e6:.1f} MB)")
    else:
        # KaggleApi reads credentials from KAGGLE_CONFIG_DIR/kaggle.json.
        os.environ["KAGGLE_CONFIG_DIR"] = str(PROJECT_ROOT)
        from kaggle.api.kaggle_api_extended import KaggleApi

        api = KaggleApi()
        api.authenticate()
        log.info(f"Downloading {DATASET_SLUG} from Kaggle...")
        api.dataset_download_files(DATASET_SLUG, path=str(RAW_DIR), unzip=True)
        log.info(f"Cached to {cache_path}")

    rows = []
    with cache_path.open("r", encoding="utf-8") as fh:
        for line in fh:
            rows.append(json.loads(line))
    df = pd.DataFrame(rows)
    log.info(f"Loaded {len(df):,} articles | columns: {list(df.columns)}")
    return df


raw_df = load_dataset()
raw_df.head()

## 2. Preprocessing — S1-T6

Calls `clean_text()` from `src/preprocessing.py` (kept as a module so it can be unit-tested in CI). Adds three numeric features (word / char / punct counts) and writes the cleaned dataset to `data/processed/cleaned.parquet`.

In [ ]:
"""S1-T6 — Apply clean_text + numeric features over the dataset.

Reads `raw_df` from the cell above, produces the cleaned dataset with
the cleaned text column plus three numeric features
(word_count, char_count, punct_count), and persists to
`data/processed/cleaned.parquet`. Cached on subsequent runs.

NOTE (added at sprint 2): also stores `raw_text` — a *lightly* cleaned
version (URL + whitespace normalisation only) — alongside `text` (the
heavily cleaned version used by classical TF-IDF). The fine-tuned
RoBERTa was trained on lightly-cleaned text, so it expects that input
format at inference time (S2-T8, S2-T12, S2-T13).
"""

import re

from tqdm.auto import tqdm

from src.preprocessing import build_numeric_features, clean_text

CLEANED_PATH = PROCESSED_DIR / "cleaned.parquet"

_URL_RE = re.compile(r"http\S+|www\S+")
_WS_RE = re.compile(r"\s+")


def _light_clean(text: str) -> str:
    """Lightweight clean — matches the format the fine-tuned RoBERTa was trained on."""
    if not text:
        return ""
    text = _URL_RE.sub("", str(text))
    text = _WS_RE.sub(" ", text)
    return text.strip()


if CLEANED_PATH.exists() and "raw_text" in pd.read_parquet(CLEANED_PATH, columns=None).columns:
    log.info(f"Using cached cleaned dataset (with raw_text) at {CLEANED_PATH}")
    cleaned_df = pd.read_parquet(CLEANED_PATH)
else:
    log.info(f"Cleaning {len(raw_df):,} articles (~30-60s)...")
    combined_raw = (
        raw_df["headline"].fillna("").map(_light_clean)
        + " "
        + raw_df["short_description"].fillna("").map(_light_clean)
    ).str.strip().tolist()  # raw text, lightly cleaned (RoBERTa-friendly)
    combined_for_clean = (
        raw_df["headline"].fillna("") + " " + raw_df["short_description"].fillna("")
    ).tolist()  # source for heavy clean + numeric features

    feats = [build_numeric_features(t) for t in tqdm(combined_for_clean, desc="numeric")]
    texts = [clean_text(t) for t in tqdm(combined_for_clean, desc="clean")]

    cleaned_df = pd.DataFrame(
        {
            "text": texts,
            "raw_text": combined_raw,
            "category": raw_df["category"].values,
            "word_count": [f["word_count"] for f in feats],
            "char_count": [f["char_count"] for f in feats],
            "punct_count": [f["punct_count"] for f in feats],
        }
    )
    cleaned_df.to_parquet(CLEANED_PATH, compression="snappy")
    log.info(f"Saved cleaned dataset to {CLEANED_PATH}")

log.info(f"Cleaned dataset: {len(cleaned_df):,} rows x {cleaned_df.shape[1]} columns ({list(cleaned_df.columns)})")
cleaned_df.head()

## 2.5. Category Consolidation — S2-T1

Per **ADR-010**, the raw 42 labels in the dataset include several near-duplicates (`STYLE` vs `STYLE & BEAUTY`, the three `ARTS` variants, etc.). Sprint 1's confusion matrix shows these are the dominant source of error. Consolidating them to **27 final classes** raises classical models from ~55% to ~66% accuracy and fine-tuned RoBERTa to ~70%.

This cell applies the merge map, drops rows whose cleaned text is shorter than 4 words, drops exact duplicates, and overwrites `data/processed/cleaned.parquet` with the 27-class dataset. All sprint-2 models train against this consolidated label space.

In [ ]:
"""S2-T1 — Category consolidation 42 → 27, drop short / duplicate rows.

Re-saves `data/processed/cleaned.parquet` in place. All downstream cells
(features, classical training, RoBERTa eval, EDA part 2, Gradio) operate
on this 27-class dataset.
"""

CATEGORY_MAP = {
    "STYLE & BEAUTY": "STYLE",
    "THE WORLDPOST": "POLITICS_WORLD",
    "WORLDPOST": "POLITICS_WORLD",
    "POLITICS": "POLITICS_WORLD",
    "WELLNESS": "HEALTH",
    "HEALTHY LIVING": "HEALTH",
    "TASTE": "FOOD",
    "FOOD & DRINK": "FOOD",
    "ARTS": "ARTS",
    "CULTURE & ARTS": "ARTS",
    "ARTS & CULTURE": "ARTS",
    "COLLEGE": "EDUCATION",
    "EDUCATION": "EDUCATION",
    "SCIENCE": "SCI_TECH",
    "TECH": "SCI_TECH",
    "GREEN": "ENVIRONMENT",
    "ENVIRONMENT": "ENVIRONMENT",
    "MONEY": "BUSINESS",
    "BUSINESS": "BUSINESS",
    "WOMEN": "FAMILY",
    "PARENTS": "FAMILY",
    "PARENTING": "FAMILY",
    "CRIME": "CRIME",
    "WEIRD NEWS": "CRIME",
    "MEDIA": "ENTERTAINMENT",
    "ENTERTAINMENT": "ENTERTAINMENT",
    "STYLE": "STYLE",
}
KEEP_AS_IS = {
    "SPORTS", "TRAVEL", "RELIGION", "COMEDY", "QUEER VOICES",
    "BLACK VOICES", "DIVORCE", "FIFTY", "GOOD NEWS", "HOME & LIVING",
    "IMPACT", "LATINO VOICES", "U.S. NEWS", "WEDDINGS", "WORLD NEWS",
}
MIN_SAMPLES_PER_CLASS = 1000
MIN_WORDS = 4


def _consolidate(cat: str) -> str | None:
    if cat in CATEGORY_MAP:
        return CATEGORY_MAP[cat]
    if cat in KEEP_AS_IS:
        return cat
    return None  # drop unmapped


log.info(f"Before consolidation: {cleaned_df['category'].nunique()} categories, {len(cleaned_df):,} rows")

# 1. Map labels.
cleaned_df = cleaned_df.assign(category=cleaned_df["category"].map(_consolidate)).dropna(subset=["category"])

# 2. Drop short cleaned texts.
n_before = len(cleaned_df)
cleaned_df = cleaned_df[cleaned_df["text"].str.split().str.len() >= MIN_WORDS]
log.info(f"Dropped {n_before - len(cleaned_df):,} rows with <{MIN_WORDS}-word cleaned text")

# 3. Drop exact duplicates.
n_before = len(cleaned_df)
cleaned_df = cleaned_df.drop_duplicates(subset=["text", "category"])
log.info(f"Dropped {n_before - len(cleaned_df):,} exact duplicates")

# 4. Drop classes below the minimum sample count (defensive).
counts = cleaned_df["category"].value_counts()
small = counts[counts < MIN_SAMPLES_PER_CLASS].index.tolist()
if small:
    cleaned_df = cleaned_df[~cleaned_df["category"].isin(small)]
    log.warning(f"Dropped {len(small)} classes below {MIN_SAMPLES_PER_CLASS}: {small}")

cleaned_df = cleaned_df.reset_index(drop=True)
cleaned_df.to_parquet(CLEANED_PATH, compression="snappy")
log.info(
    f"After consolidation: {cleaned_df['category'].nunique()} categories, "
    f"{len(cleaned_df):,} rows. Saved to {CLEANED_PATH}"
)
cleaned_df["category"].value_counts()

## 2.6. Download Fine-tuned RoBERTa — S2-T2

Per **ADR-011** (supersedes ADR-003), the project's primary classifier is a `RobertaForSequenceClassification` model fine-tuned on the 27-class merged dataset, sourced from the team coordinator's prior individual work (see `docs/decisions.md` for full attribution).

The model package lives on Google Drive (**ADR-012**) at FILE_ID `19EIWqmmR4tbJrMyiqKYRT__s_d1n11rW` and weighs ~470 MB. First call downloads + extracts it; subsequent calls reuse the local cache under `models/best_model/`.

In [ ]:
"""S2-T2 — Download (or reuse cached) fine-tuned RoBERTa, then load it.

Network call only on the first run. After extraction, `models/best_model/`
contains config.json, model.safetensors, tokenizer.json,
tokenizer_config.json, training_args.bin.
"""

import subprocess
import zipfile

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

DRIVE_FILE_ID = "19EIWqmmR4tbJrMyiqKYRT__s_d1n11rW"  # ADR-012
BEST_MODEL_DIR = MODELS_DIR / "best_model"
BEST_MODEL_ZIP = MODELS_DIR / "best_model.zip"

if (BEST_MODEL_DIR / "config.json").exists():
    log.info(f"Using cached fine-tuned model at {BEST_MODEL_DIR}")
else:
    log.info(f"Downloading best_model.zip from Drive (~470 MB) ...")
    try:
        import gdown
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
        import gdown

    gdown.download(id=DRIVE_FILE_ID, output=str(BEST_MODEL_ZIP), quiet=False)

    log.info(f"Extracting {BEST_MODEL_ZIP.name} ...")
    with zipfile.ZipFile(BEST_MODEL_ZIP) as zf:
        zf.extractall(MODELS_DIR)
    BEST_MODEL_ZIP.unlink()
    log.info(f"Extracted to {BEST_MODEL_DIR}")

# Load model + tokenizer.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL_DIR)
roberta_model = AutoModelForSequenceClassification.from_pretrained(BEST_MODEL_DIR).to(device).eval()
log.info(
    f"Loaded {roberta_model.__class__.__name__} | "
    f"num_labels={roberta_model.config.num_labels} | device={device}"
)
assert roberta_model.config.num_labels == 27, (
    f"Expected 27 labels, got {roberta_model.config.num_labels}"
)

# Sanity check on 5 random samples.
sample = cleaned_df.sample(5, random_state=RANDOM_STATE)[["text", "category"]].reset_index(drop=True)
with torch.no_grad():
    enc = tokenizer(
        sample["text"].tolist(),
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt",
    ).to(device)
    logits = roberta_model(**enc).logits
    pred_ids = logits.argmax(dim=-1).cpu().numpy()
predicted = [roberta_model.config.id2label[int(i)] for i in pred_ids]
print("Sanity check (5 random samples):")
for (_, row), pred in zip(sample.iterrows(), predicted):
    mark = "✓" if pred == row.category else " "
    print(f"  {mark} true=[{row.category:>15}]  pred=[{pred:>15}]  | {row.text[:60]}")

## 3. EDA

Exploratory analysis lives in `notebooks/01_eda.ipynb`. Sprint 1 produces part 1 (shape / missing / class distribution); sprint 2 adds the full chart set.

## 4. Classical Features — S1-T8

TF-IDF (uni+bi-grams, capped at 50K features, sublinear TF) stacked with three numeric features (word / char / punct counts, scaled). Persisted to `models/`.

In [ ]:
"""S1-T8 — TF-IDF (uni+bi-gram, 50K cap, sublinear, min_df=2) + numeric features.

Stratified 80/20 split with `random_state=RANDOM_STATE`, vectorizer fit on
TRAIN ONLY (PRD F4), scaler fit on TRAIN ONLY. Persists vectorizer,
numeric scaler and label encoder to `models/`. Includes the round-trip
test (fit → save → load → transform produces an identical matrix).

Outputs in scope: X_train, X_test, y_train, y_test, plus the raw text
arrays for downstream display. Subsequent cells (S1-T9 train, S1-T10
evaluate) reuse these.

Sprint-2 addition: also splits `raw_text` (lightly-cleaned text used by
the fine-tuned RoBERTa in S2-T8/T12/T13) so it stays aligned with
y_test.
"""

import joblib
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Encode labels -> integer ids; persist for downstream inference.
label_encoder = LabelEncoder()
y_all = label_encoder.fit_transform(cleaned_df["category"].values)
joblib.dump(label_encoder, MODELS_DIR / "label_encoder.joblib")
log.info(f"Label encoder fit on {len(label_encoder.classes_)} classes; saved.")

# Stratified 80/20 split — same indices reused for every model in sprint 2.
(
    X_text_train, X_text_test,
    X_num_train, X_num_test,
    X_raw_train, X_raw_test,
    y_train, y_test,
) = train_test_split(
    cleaned_df["text"].values,
    cleaned_df[["word_count", "char_count", "punct_count"]].to_numpy(),
    cleaned_df["raw_text"].values,
    y_all,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_all,
)
log.info(f"Train: {len(X_text_train):,}   Test: {len(X_text_test):,}")

# Fit TF-IDF on TRAIN only, transform both.
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50_000,
    sublinear_tf=True,
    min_df=2,
)
X_tfidf_train = tfidf.fit_transform(X_text_train)
X_tfidf_test = tfidf.transform(X_text_test)
joblib.dump(tfidf, MODELS_DIR / "tfidf_vectorizer.joblib")
log.info(f"TF-IDF vocab: {len(tfidf.vocabulary_):,} features")

# Fit StandardScaler on TRAIN numeric only, transform both.
scaler = StandardScaler()
X_num_train_scaled = scaler.fit_transform(X_num_train)
X_num_test_scaled = scaler.transform(X_num_test)
joblib.dump(scaler, MODELS_DIR / "numeric_scaler.joblib")

# Sparse-stack TF-IDF + scaled numeric (3 cols).
X_train = hstack([X_tfidf_train, csr_matrix(X_num_train_scaled)]).tocsr()
X_test = hstack([X_tfidf_test, csr_matrix(X_num_test_scaled)]).tocsr()
log.info(f"X_train: {X_train.shape}   X_test: {X_test.shape}   nnz/row: {X_train.nnz / X_train.shape[0]:.1f}")

# Round-trip test (S1-T8 acceptance).
tfidf_reload = joblib.load(MODELS_DIR / "tfidf_vectorizer.joblib")
scaler_reload = joblib.load(MODELS_DIR / "numeric_scaler.joblib")
sample_idx = slice(0, 5)
roundtrip = hstack(
    [tfidf_reload.transform(X_text_test[sample_idx]), csr_matrix(scaler_reload.transform(X_num_test[sample_idx]))]
).tocsr()
diff = X_test[sample_idx] - roundtrip
assert diff.nnz == 0, f"Round-trip mismatch: {diff.nnz} non-zero deltas"
log.info("Round-trip test passed: load(saved).transform == original.transform on 5 sample rows")

## 5. Train Logistic Regression Baseline — S1-T9

Stratified 80/20 split, `class_weight='balanced'`, `GridSearchCV` over `C ∈ {0.1, 1, 10}`. Persists best estimator to `models/logreg_best.joblib`.

In [ ]:
"""S1-T9 — LogReg baseline with GridSearchCV.

Logistic Regression with `class_weight='balanced'`, multinomial via the
`saga` solver (good for large sparse text features), grid search over
`C ∈ {0.1, 1, 10}` with 3-fold CV scoring on macro-F1.

`saga` chosen over `lbfgs` for memory/speed on sparse 50K-dim input;
over `liblinear` because we want multinomial probabilities, not OvR.
Persists the best refit-on-full-train estimator to
`models/logreg_best.joblib`.
"""

import time

import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV

base = LogisticRegression(
    class_weight="balanced",
    solver="saga",
    max_iter=300,
    random_state=RANDOM_STATE,
    n_jobs=1,  # let GridSearchCV parallelise across folds/params
)

search = GridSearchCV(
    base,
    param_grid={"C": [0.1, 1.0, 10.0]},
    cv=3,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1,
)

t0 = time.time()
log.info("Starting GridSearchCV(cv=3) over C in {0.1, 1.0, 10.0} ...")
search.fit(X_train, y_train)
log.info(f"Done in {(time.time() - t0) / 60:.1f} min")
log.info(f"Best C: {search.best_params_['C']}, mean CV macro-F1: {search.best_score_:.4f}")

logreg_best = search.best_estimator_
joblib.dump(logreg_best, MODELS_DIR / "logreg_best.joblib")

# Quick test-set sanity check (full evaluation suite runs in S1-T10).
y_pred = logreg_best.predict(X_test)
test_macro_f1 = f1_score(y_test, y_pred, average="macro")
test_weighted_f1 = f1_score(y_test, y_pred, average="weighted")
log.info(f"Test set: macro-F1 = {test_macro_f1:.4f} | weighted-F1 = {test_weighted_f1:.4f}")
assert test_macro_f1 >= 0.45, f"S1-T9 acceptance fail: macro-F1 {test_macro_f1:.4f} < 0.45"
log.info("S1-T9 acceptance met (macro-F1 >= 0.45).")

## 6. Evaluate Models — S1-T10

Computes (accuracy, precision, recall, F1-macro, F1-weighted, ROC-AUC OvR) for each persisted model and appends a row to `reports/model_comparison.csv`. Saves a confusion-matrix PNG to `reports/confusion/<name>.png`.

In [ ]:
"""S1-T10 — `evaluate_model()` skeleton + first row in comparison table.

Computes (accuracy, precision_macro, recall_macro, f1_macro, f1_weighted,
roc_auc_ovr_macro) for any fitted sklearn classifier. Appends a row to
`reports/model_comparison.csv` (replacing any prior row for the same model)
and saves a row-normalised confusion-matrix PNG to
`reports/confusion/<name>.png`.

Sprint 2's models call this same function — that's why the per-model
row-replacement logic exists.
"""

from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

COMPARISON_CSV = REPORTS_DIR / "model_comparison.csv"


def evaluate_model(model, name, X_test, y_test, label_encoder=None):
    """Compute metrics + persist comparison row + confusion-matrix PNG."""
    y_pred = model.predict(X_test)

    metrics = {
        "model": name,
        "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    }
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test)
        metrics["roc_auc_ovr"] = roc_auc_score(y_test, proba, multi_class="ovr", average="macro")
    else:
        metrics["roc_auc_ovr"] = float("nan")

    # Append/replace row in comparison CSV.
    new_row = pd.DataFrame([metrics])
    if COMPARISON_CSV.exists():
        existing = pd.read_csv(COMPARISON_CSV)
        existing = existing[existing["model"] != name]
        out = pd.concat([existing, new_row], ignore_index=True)
    else:
        out = new_row
    out.to_csv(COMPARISON_CSV, index=False)
    log.info(f"Wrote '{name}' row to {COMPARISON_CSV.name} ({len(out)} model rows total)")

    # Confusion matrix.
    classes = label_encoder.classes_ if label_encoder is not None else np.unique(y_test)
    cm = confusion_matrix(y_test, y_pred, normalize="true")
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(
        cm,
        ax=ax,
        cmap="Blues",
        vmin=0,
        vmax=1,
        xticklabels=classes,
        yticklabels=classes,
        square=True,
        cbar_kws={"shrink": 0.7},
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"Confusion matrix (row-normalised) — {name}")
    plt.xticks(rotation=90, fontsize=7)
    plt.yticks(fontsize=7)
    plt.tight_layout()
    cm_path = CONFUSION_DIR / f"{name}.png"
    fig.savefig(cm_path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    log.info(f"Saved confusion matrix to {cm_path}")

    return metrics


# Run on the LogReg baseline.
metrics = evaluate_model(logreg_best, "logreg", X_test, y_test, label_encoder)
print(f"LogReg baseline:")
for k, v in metrics.items():
    fmt = f"{v:.4f}" if isinstance(v, float) else str(v)
    print(f"  {k:<18}: {fmt}")

## 5.5. Sprint 2 — 5 Additional Classical Models (S2-T3 to S2-T7)

The remaining five classical models from spec §F5 (Linear SVM, KNN, Decision Tree, Random Forest, AdaBoost), each trained with cross-validated hyper-parameter tuning on the 27-class merged dataset. All call `evaluate_model()` from S1-T10 to append a row to `reports/model_comparison.csv` and save a confusion-matrix PNG.

CV folds are reduced to 2-3 (vs the full 5 in some literature) because Colab T4 free-tier has only 2 vCPUs and 6 model × 5-fold tuning would not complete in a single session. The grids below are calibrated to the 30-60 min per-model budget that fits Colab's idle disconnect window.

In [ ]:
"""S2-T3 — Linear SVM with GridSearchCV(C ∈ {0.1, 1, 10}) + calibration.

Two-stage approach for speed: tune LinearSVC alone (fast), then wrap
the best estimator in CalibratedClassifierCV(cv=3) to expose
predict_proba for the Gradio confidence display (per ADR-008).
"""

import time

from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC

# Stage 1: tune LinearSVC alone
log.info("S2-T3 stage 1: GridSearchCV(cv=3) over LinearSVC C ∈ {0.1, 1, 10} ...")
t0 = time.time()
search_svm = GridSearchCV(
    LinearSVC(class_weight="balanced", random_state=RANDOM_STATE, max_iter=2000, dual="auto"),
    param_grid={"C": [0.1, 1.0, 10.0]},
    cv=3,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1,
)
search_svm.fit(X_train, y_train)
log.info(f"Tuning done in {(time.time() - t0) / 60:.1f} min | best C: {search_svm.best_params_['C']} | CV macro-F1: {search_svm.best_score_:.4f}")

# Stage 2: wrap best LinearSVC in CalibratedClassifierCV
log.info("S2-T3 stage 2: calibrating best LinearSVC for predict_proba ...")
t0 = time.time()
linsvm_best = CalibratedClassifierCV(
    LinearSVC(
        C=search_svm.best_params_["C"],
        class_weight="balanced",
        random_state=RANDOM_STATE,
        max_iter=2000,
        dual="auto",
    ),
    cv=3,
    n_jobs=-1,
)
linsvm_best.fit(X_train, y_train)
log.info(f"Calibration done in {(time.time() - t0) / 60:.1f} min")

joblib.dump(linsvm_best, MODELS_DIR / "linearsvc_best.joblib")
metrics = evaluate_model(linsvm_best, "linearsvc", X_test, y_test, label_encoder)
print(f"LinearSVM:  acc={metrics['accuracy']:.4f}  macro-F1={metrics['f1_macro']:.4f}  weighted-F1={metrics['f1_weighted']:.4f}  ROC-AUC={metrics['roc_auc_ovr']:.4f}")

In [ ]:
"""S2-T4 — KNN with RandomizedSearchCV(n_neighbors, weights), cosine metric.

KNN scales poorly to 165K-row × 50K-feature problems — inference is the
bottleneck because every test point needs distances to every train
point. Reduced n_iter and cv=2 keep this within the per-task time budget.
The comparison-table commentary will note that KNN underperforms on
sparse high-dim text.
"""

from sklearn.model_selection import RandomizedSearchCV
from sklearn.neighbors import KNeighborsClassifier

t0 = time.time()
log.info("S2-T4: KNN with RandomizedSearchCV(n_iter=4, cv=2, cosine metric) ...")
search_knn = RandomizedSearchCV(
    KNeighborsClassifier(metric="cosine", n_jobs=-1),
    param_distributions={
        "n_neighbors": [3, 5, 7, 11, 21],
        "weights": ["uniform", "distance"],
    },
    n_iter=4,
    cv=2,
    scoring="f1_macro",
    n_jobs=1,  # KNN is already parallel via n_jobs in the estimator
    random_state=RANDOM_STATE,
    verbose=1,
)
search_knn.fit(X_train, y_train)
log.info(f"Done in {(time.time() - t0) / 60:.1f} min | best: {search_knn.best_params_}")

knn_best = search_knn.best_estimator_
joblib.dump(knn_best, MODELS_DIR / "knn_best.joblib")
metrics = evaluate_model(knn_best, "knn", X_test, y_test, label_encoder)
print(f"KNN:        acc={metrics['accuracy']:.4f}  macro-F1={metrics['f1_macro']:.4f}  weighted-F1={metrics['f1_weighted']:.4f}")

In [ ]:
"""S2-T5 — Decision Tree with GridSearchCV(max_depth, min_samples_leaf)."""

from sklearn.tree import DecisionTreeClassifier

t0 = time.time()
log.info("S2-T5: Decision Tree with GridSearchCV(cv=2) ...")
search_dt = GridSearchCV(
    DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
    param_grid={
        "max_depth": [10, 30, None],
        "min_samples_leaf": [1, 5, 20],
    },
    cv=2,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1,
)
search_dt.fit(X_train, y_train)
log.info(f"Done in {(time.time() - t0) / 60:.1f} min | best: {search_dt.best_params_}")

dt_best = search_dt.best_estimator_
log.info(f"Tree depth: {dt_best.get_depth()} | leaves: {dt_best.get_n_leaves():,}")
joblib.dump(dt_best, MODELS_DIR / "decision_tree_best.joblib")
metrics = evaluate_model(dt_best, "decision_tree", X_test, y_test, label_encoder)
print(f"Decision Tree: acc={metrics['accuracy']:.4f}  macro-F1={metrics['f1_macro']:.4f}  weighted-F1={metrics['f1_weighted']:.4f}")

In [ ]:
"""S2-T6 — Random Forest tuned on a 30K subsample, refit on full train.

Tuning on the full 165K × 50K sparse data is too slow on Colab T4
(estimated 60+ min per fit). The standard mitigation per R4 in the
risk register: tune on a stratified subsample, then refit best params
on the full training set.
"""

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample

# Stratified subsample for tuning
n_sub = 30_000
log.info(f"S2-T6 stage 1: tune on stratified {n_sub:,}-row subsample ...")
idx_sub = resample(
    np.arange(X_train.shape[0]),
    n_samples=n_sub,
    replace=False,
    stratify=y_train,
    random_state=RANDOM_STATE,
)
X_sub = X_train[idx_sub]
y_sub = y_train[idx_sub]

t0 = time.time()
search_rf = RandomizedSearchCV(
    RandomForestClassifier(
        class_weight="balanced",
        oob_score=True,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    param_distributions={
        "n_estimators": [100, 200],
        "max_depth": [20, None],
    },
    n_iter=3,
    cv=2,
    scoring="f1_macro",
    n_jobs=1,
    random_state=RANDOM_STATE,
    verbose=1,
)
search_rf.fit(X_sub, y_sub)
log.info(f"Tuning done in {(time.time() - t0) / 60:.1f} min | best: {search_rf.best_params_}")

# Refit best params on full train
log.info("S2-T6 stage 2: refit best RF on full train set ...")
t0 = time.time()
rf_best = RandomForestClassifier(
    **search_rf.best_params_,
    class_weight="balanced",
    oob_score=True,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
rf_best.fit(X_train, y_train)
log.info(f"Refit done in {(time.time() - t0) / 60:.1f} min | OOB score: {rf_best.oob_score_:.4f}")

joblib.dump(rf_best, MODELS_DIR / "rf_best.joblib")
metrics = evaluate_model(rf_best, "random_forest", X_test, y_test, label_encoder)
print(f"Random Forest: acc={metrics['accuracy']:.4f}  macro-F1={metrics['f1_macro']:.4f}  weighted-F1={metrics['f1_weighted']:.4f}")

In [ ]:
"""S2-T7 — AdaBoost with GridSearchCV(n_estimators, learning_rate)."""

from sklearn.ensemble import AdaBoostClassifier

t0 = time.time()
log.info("S2-T7: AdaBoost with GridSearchCV(cv=2, SAMME) ...")
search_ada = GridSearchCV(
    AdaBoostClassifier(random_state=RANDOM_STATE, algorithm="SAMME"),
    param_grid={
        "n_estimators": [50, 100],
        "learning_rate": [0.5, 1.0],
    },
    cv=2,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1,
)
search_ada.fit(X_train, y_train)
log.info(f"Done in {(time.time() - t0) / 60:.1f} min | best: {search_ada.best_params_}")

ada_best = search_ada.best_estimator_
joblib.dump(ada_best, MODELS_DIR / "adaboost_best.joblib")
metrics = evaluate_model(ada_best, "adaboost", X_test, y_test, label_encoder)
print(f"AdaBoost:   acc={metrics['accuracy']:.4f}  macro-F1={metrics['f1_macro']:.4f}  weighted-F1={metrics['f1_weighted']:.4f}")

## 7. Evaluate Fine-tuned RoBERTa — S2-T8

Inference over the held-out test set with the fine-tuned model loaded in §2.6 (S2-T2). Runs in batches under `torch.no_grad()`, computes the standard metric suite from `evaluate_model()`, and adds a **`roberta_finetuned`** row to `reports/model_comparison.csv` plus a confusion-matrix PNG.

The RoBERTa model's internal `id2label` ordering may differ from `LabelEncoder.classes_` ordering; we remap predictions before computing metrics against `y_test`.

In [ ]:
"""S2-T8 — Evaluate fine-tuned RoBERTa on the held-out test set.

Uses **`X_raw_test`** (lightly-cleaned text — URL strip + whitespace
normalisation only) because that's the format the fine-tuned model was
trained on. Heavy NLTK-cleaned `X_text_test` causes a large accuracy
drop because the tokenizer sees out-of-distribution input.
"""

from datetime import datetime, timezone

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Build label-id remap between RoBERTa's internal ordering and our LabelEncoder.
encoder_label2id = {label: i for i, label in enumerate(label_encoder.classes_)}
roberta_id2label = {int(k): v for k, v in roberta_model.config.id2label.items()}
remap = np.array([encoder_label2id[roberta_id2label[i]] for i in range(roberta_model.config.num_labels)])

# Predict in batches over the LIGHTLY CLEANED raw text (matches training format).
device_t = roberta_model.device
log.info(f"S2-T8: predicting on {len(X_raw_test):,} test rows with fine-tuned RoBERTa (raw_text input) ...")
t0 = time.time()
BATCH = 64
all_probs_roberta_id = []
with torch.no_grad():
    for i in range(0, len(X_raw_test), BATCH):
        batch_texts = X_raw_test[i : i + BATCH].tolist()
        enc = tokenizer(
            batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt"
        ).to(device_t)
        logits = roberta_model(**enc).logits
        all_probs_roberta_id.append(F.softmax(logits, dim=-1).cpu().numpy())
y_prob_roberta = np.concatenate(all_probs_roberta_id, axis=0)  # ordered by RoBERTa's internal ids
log.info(f"Inference done in {(time.time() - t0) / 60:.1f} min")

# Remap proba columns from RoBERTa-id space → LabelEncoder-id space.
y_prob = np.zeros_like(y_prob_roberta)
for r_id, e_id in enumerate(remap):
    y_prob[:, e_id] = y_prob_roberta[:, r_id]
y_pred = y_prob.argmax(axis=1)

# Metrics.
metrics = {
    "model": "roberta_finetuned",
    "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "accuracy": accuracy_score(y_test, y_pred),
    "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    "roc_auc_ovr": roc_auc_score(y_test, y_prob, multi_class="ovr", average="macro"),
}

# Append/replace row in comparison CSV.
new_row = pd.DataFrame([metrics])
existing = pd.read_csv(COMPARISON_CSV)
existing = existing[existing["model"] != "roberta_finetuned"]
out = pd.concat([existing, new_row], ignore_index=True)
out.to_csv(COMPARISON_CSV, index=False)
log.info(f"Wrote roberta_finetuned row to {COMPARISON_CSV.name}")

# Confusion matrix PNG.
classes = label_encoder.classes_
cm = confusion_matrix(y_test, y_pred, normalize="true")
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cm, ax=ax, cmap="Blues", vmin=0, vmax=1,
    xticklabels=classes, yticklabels=classes, square=True, cbar_kws={"shrink": 0.7},
)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion matrix (row-normalised) — roberta_finetuned")
plt.xticks(rotation=90, fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
fig.savefig(CONFUSION_DIR / "roberta_finetuned.png", dpi=120, bbox_inches="tight")
plt.close(fig)

print("Fine-tuned RoBERTa metrics:")
for k, v in metrics.items():
    fmt = f"{v:.4f}" if isinstance(v, float) else str(v)
    print(f"  {k:<18}: {fmt}")
assert metrics["accuracy"] >= 0.65, f"S2-T8 acceptance fail: accuracy {metrics['accuracy']:.4f} < 0.65"
log.info("S2-T8 acceptance met (accuracy >= 0.65, expected ≈ 0.70).")

## 7.5. Finalise Comparison Table — S2-T9

All seven models trained: LogReg (sprint 1) + LinearSVC + KNN + Decision Tree + Random Forest + AdaBoost (sprint 2) + Fine-tuned RoBERTa. This cell loads `reports/model_comparison.csv`, sorts by macro-F1 descending, and renders the styled table with the best-row highlighted. Confirms the comparison file has 7 rows × 7 metric columns.

In [ ]:
"""S2-T9 — Finalise comparison table + render styled view."""

comparison = pd.read_csv(COMPARISON_CSV).sort_values("f1_macro", ascending=False).reset_index(drop=True)
comparison.to_csv(COMPARISON_CSV, index=False)

assert len(comparison) == 7, f"Expected 7 rows in comparison, got {len(comparison)}"
log.info(f"Comparison: {len(comparison)} rows × {comparison.shape[1]} columns")
print(f"Best model: **{comparison.iloc[0]['model']}** (macro-F1 = {comparison.iloc[0]['f1_macro']:.4f})")

# Constants used by the Gradio app + report.
BEST_MODEL_NAME = comparison.iloc[0]["model"]
log.info(f"BEST_MODEL_NAME = {BEST_MODEL_NAME!r}")

# Styled DataFrame with best row highlighted.
numeric_cols = ["accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_weighted", "roc_auc_ovr"]
styled = (
    comparison[["model"] + numeric_cols]
    .style
    .format({c: "{:.4f}" for c in numeric_cols})
    .highlight_max(subset=numeric_cols, color="#90EE90")
    .background_gradient(subset=numeric_cols, cmap="Greens", vmin=0, vmax=1)
    .set_caption(f"Sprint 2 model comparison — best: {BEST_MODEL_NAME}")
)
styled

## 8. Groq LLM — Real Integration with Caching — S2-T11

Replaces the sprint-1 smoke test with the real `explain(text, label, confidence)` function used by the Gradio app. Uses the spec's prompt template, ADR-001's `llama-3.3-70b-versatile`, with `lru_cache` keyed by `(sha1(text)[:16], label)` so repeated identical queries (common during the live demo) hit cache. Failures (rate-limit, network) return a graceful placeholder — the UI never crashes.

In [ ]:
"""S2-T11 — Real Groq LLM integration: explain(text, label, confidence) -> str.

Cached via lru_cache by (sha1(text)[:16], label). Failures return a
friendly placeholder; the function never raises.
"""

import hashlib
from functools import lru_cache

GROQ_MODEL = "llama-3.3-70b-versatile"  # ADR-001
GROQ_KEY = os.environ.get("GROQ_API_KEY")

PROMPT_TEMPLATE = (
    "You are a concise news-classification assistant. An automatic classifier "
    "predicted a category for the article below. In 2-3 sentences, explain "
    "why the category fits, focusing on the article's key topics and themes. "
    "Be specific to the content; do not restate the category name only.\n\n"
    "Article: {article}\n"
    "Predicted category: {category}\n"
    "Confidence: {confidence:.0%}\n\n"
    "Explanation:"
)


def _text_hash(text: str) -> str:
    return hashlib.sha1(text.encode("utf-8")).hexdigest()[:16]


@lru_cache(maxsize=1024)
def _explain_cached(text_hash_key: str, label: str, confidence_centi: int, text: str) -> str:
    """Inner cached function. text_hash_key is the cache discriminator."""
    if not GROQ_KEY:
        return "(LLM disabled — set GROQ_API_KEY in Colab Secrets to enable explanations.)"
    try:
        from groq import Groq

        client = Groq(api_key=GROQ_KEY)
        prompt = PROMPT_TEMPLATE.format(
            article=text[:1000],  # cap context length
            category=label,
            confidence=confidence_centi / 100.0,
        )
        completion = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200,
            temperature=0.3,
        )
        return completion.choices[0].message.content.strip()
    except Exception as e:
        log.warning(f"Groq call failed: {type(e).__name__}: {e}")
        return "(LLM explanation temporarily unavailable.)"


def explain(text: str, label: str, confidence: float) -> str:
    """Public entry: returns 2-3 sentence explanation of why `label` fits `text`."""
    return _explain_cached(_text_hash(text), label, int(round(confidence * 100)), text)


# Smoke test — fires one real call per unique (text, label) input.
sample_explanation = explain(
    "Apple unveils new iPhone with improved camera and battery life",
    "SCI_TECH",
    0.85,
)
print(f"Sample explanation:\n{sample_explanation}")
print(f"\nCache info: {_explain_cached.cache_info()}")

## 8.5. Similarity Retriever — S2-T12

Reuses the **fine-tuned RoBERTa encoder** (the `.roberta` sub-module of `roberta_model`, which is just the base RoBERTa transformer without the classification head) to produce a 768-d embedding per article. Embeds the entire training set in batches under `torch.no_grad()`, mean-pools last-hidden-states masked by `attention_mask`, caches `(N, 768)` `float32` to `data/embeddings/roberta_finetuned.npy` plus a parallel `index.parquet` with `(text, category)`.

`top_k(query, k=3)` then encodes the query the same way and ranks the cached corpus by cosine similarity.

**First run:** ~30-45 min on Colab T4 for the corpus pass. **Subsequent runs:** instant cache load.

In [ ]:
"""S2-T12 — Similarity retriever via the fine-tuned RoBERTa encoder.

Uses **`X_raw_train`** (lightly-cleaned text) — same input format the
encoder was trained on — to compute and cache the corpus embeddings.
"""

import time

import numpy as np
import torch
from sklearn.metrics.pairwise import cosine_similarity

EMBEDDINGS_NPY = EMBEDDINGS_DIR / "roberta_finetuned.npy"
INDEX_PARQUET = EMBEDDINGS_DIR / "index.parquet"

if EMBEDDINGS_NPY.exists() and INDEX_PARQUET.exists():
    log.info(f"Loading cached embeddings from {EMBEDDINGS_NPY.name}")
    train_embeddings = np.load(EMBEDDINGS_NPY)
    train_index = pd.read_parquet(INDEX_PARQUET)
else:
    log.info(f"Embedding {len(X_raw_train):,} train texts with fine-tuned encoder ...")
    encoder = roberta_model.roberta  # base transformer, no classification head
    chunks = []
    BATCH = 64
    t0 = time.time()
    with torch.no_grad():
        for i in range(0, len(X_raw_train), BATCH):
            batch = X_raw_train[i : i + BATCH].tolist()
            enc = tokenizer(
                batch, padding=True, truncation=True, max_length=128, return_tensors="pt"
            ).to(roberta_model.device)
            out = encoder(**enc).last_hidden_state  # (B, T, 768)
            mask = enc["attention_mask"].unsqueeze(-1).float()
            pooled = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            chunks.append(pooled.cpu().numpy().astype(np.float32))
            if i % (BATCH * 100) == 0 and i > 0:
                done_min = (time.time() - t0) / 60
                eta_min = done_min * (len(X_raw_train) - i) / i
                log.info(f"  {i:,}/{len(X_raw_train):,} | elapsed {done_min:.1f} min | ETA {eta_min:.1f} min")
    train_embeddings = np.concatenate(chunks, axis=0)
    train_index = pd.DataFrame({
        "headline": X_raw_train,  # the lightly-cleaned headline+description
        "category": [label_encoder.inverse_transform([y])[0] for y in y_train],
    })
    np.save(EMBEDDINGS_NPY, train_embeddings)
    train_index.to_parquet(INDEX_PARQUET)
    log.info(
        f"Cached: {EMBEDDINGS_NPY.name} {train_embeddings.shape} | "
        f"total wall-time {(time.time() - t0) / 60:.1f} min"
    )


def _embed_query(text: str) -> np.ndarray:
    """Embed a single query through the fine-tuned encoder. Expects RAW (lightly-cleaned) text."""
    encoder = roberta_model.roberta
    with torch.no_grad():
        enc = tokenizer(
            [text], padding=True, truncation=True, max_length=128, return_tensors="pt"
        ).to(roberta_model.device)
        out = encoder(**enc).last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1).float()
        pooled = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
    return pooled.cpu().numpy().astype(np.float32)


def top_k(query_text: str, k: int = 3) -> list:
    """Return top-k similar training articles by cosine similarity. Pass raw / lightly-cleaned text."""
    q_emb = _embed_query(query_text)
    sims = cosine_similarity(q_emb, train_embeddings)[0]
    top_idx = np.argsort(-sims)[:k]
    return [
        {
            "headline": train_index.iloc[int(i)]["headline"],
            "category": train_index.iloc[int(i)]["category"],
            "score": float(sims[int(i)]),
        }
        for i in top_idx
    ]


# Smoke test: latency target < 200 ms (PRD §F9).
test_query = "Apple unveils new iPhone with improved camera"
t0 = time.time()
results = top_k(test_query, k=3)
ms = (time.time() - t0) * 1000
log.info(f"top_k smoke: {ms:.0f} ms")
print(f"Query: {test_query}")
print(f"Top-3 similar (latency {ms:.0f} ms):")
for i, r in enumerate(results):
    print(f"  {i + 1}. [{r['category']:>15}] (score {r['score']:.3f}) {r['headline'][:70]}")

## 9. Full Gradio App — M1 Demo — S2-T13

Final UI: two text inputs (headline + description), one Predict button, four outputs (predicted category, confidence %, LLM explanation, top-3 similar articles). The classifier is the **fine-tuned RoBERTa** (best on the comparison table); explanation comes from `explain()` (S2-T11); similarity from `top_k()` (S2-T12). All three are fault-tolerant — Groq or retrieval failures degrade gracefully without crashing the UI.

PRD §F10 acceptance: end-to-end inference < 5 s on a warm Colab GPU, public share link printed.

In [ ]:
"""S2-T13 — Full Gradio app: category + confidence + LLM explanation + top-3 similar.

Feeds **lightly-cleaned raw text** (URL strip + whitespace normalisation
only) to the fine-tuned RoBERTa, matching the training input format.
The heavy `clean_text` from `src.preprocessing` is NOT used in the
inference path because RoBERTa was trained on lightly-cleaned text.
"""

import re

import gradio as gr
import torch
import torch.nn.functional as F

_URL_RE_UI = re.compile(r"http\S+|www\S+")
_WS_RE_UI = re.compile(r"\s+")


def _light_clean_ui(text: str) -> str:
    if not text:
        return ""
    text = _URL_RE_UI.sub("", str(text))
    text = _WS_RE_UI.sub(" ", text)
    return text.strip()


def predict_news_full(headline: str, description: str):
    """Gradio backend for the M1 UI. 4 outputs."""
    if not (headline or "").strip() and not (description or "").strip():
        return "Please enter a headline or description.", 0.0, "—", "—"

    raw_input = (_light_clean_ui(headline) + " " + _light_clean_ui(description)).strip()

    # 1. Predict via fine-tuned RoBERTa over the raw input.
    with torch.no_grad():
        enc = tokenizer(
            [raw_input],
            padding=True, truncation=True, max_length=128, return_tensors="pt",
        ).to(roberta_model.device)
        logits = roberta_model(**enc).logits
        probs = F.softmax(logits, dim=-1)[0]
        top_idx = int(probs.argmax())
        confidence = float(probs[top_idx])
        category = roberta_model.config.id2label[top_idx]

    confidence_pct = round(confidence * 100, 1)

    # 2. LLM explanation (graceful fallback on error).
    try:
        explanation = explain(raw_input, category, confidence)
    except Exception as e:
        log.warning(f"LLM explanation failed in UI: {e}")
        explanation = "(LLM explanation unavailable.)"

    # 3. Top-3 similar articles (graceful fallback). Use the SAME raw input.
    try:
        similar = top_k(raw_input, k=3)
        similar_md = "\n\n".join(
            f"**{i + 1}.** _{r['category']}_  •  similarity {r['score']:.3f}  \n{r['headline'][:200]}"
            for i, r in enumerate(similar)
        )
    except Exception as e:
        log.warning(f"Similarity retrieval failed in UI: {e}")
        similar_md = "(Similar articles unavailable.)"

    return category, confidence_pct, explanation, similar_md


demo = gr.Interface(
    fn=predict_news_full,
    inputs=[
        gr.Textbox(label="Headline", placeholder="e.g. Apple unveils new iPhone with improved camera"),
        gr.Textbox(label="Short description (optional)", lines=3),
    ],
    outputs=[
        gr.Textbox(label="Predicted category"),
        gr.Number(label="Confidence (%)"),
        gr.Textbox(label="LLM explanation", lines=4),
        gr.Markdown(label="Top-3 similar articles", value=""),
    ],
    title="News Category Classification — M1 Demo",
    description=(
        "Fine-tuned RoBERTa over **27 consolidated** news categories, with a Groq Llama 3.3 70B "
        "explanation of the prediction and the **top-3 most similar articles** from the training "
        "set ranked by cosine similarity over the model's encoder embeddings."
    ),
    allow_flagging="never",
    examples=[
        ["Apple unveils new iPhone with improved camera and battery life",
         "The latest model emphasises low-light photography and a 25% larger battery."],
        ["Senate passes sweeping climate bill in late-night vote",
         "The bill commits $370B to clean energy and electric vehicle subsidies over the next decade."],
        ["Lakers beat Celtics in overtime thriller",
         "LeBron James scored 38 points as the Lakers held off a fourth-quarter Celtics rally."],
    ],
)

# In Colab, share=True produces a public link valid ~72 h.
demo.launch(share=True)